# Prediction Results Plots

Grouped plots comparing gesture and phase prediction results across tasks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 450,
    "font.size": 16.5,
    "axes.labelsize": 18,
    "axes.titlesize": 21,
    "legend.fontsize": 17,
})

In [ ]:
# Load the shared configuration from config.yaml at the repository root
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.yaml').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('config.yaml not found. Run this notebook from inside the repository.')
with open(REPO_ROOT / 'config.yaml') as f:
    CONFIG = yaml.safe_load(f)
PATHS = {name: REPO_ROOT / path for name, path in CONFIG['paths'].items()}

# Combine the results CSVs written by the training notebooks
results_files = sorted(PATHS['results_dir'].glob('*.csv'))
if not results_files:
    raise FileNotFoundError(f"No results found in {PATHS['results_dir']}. Run the training notebooks first.")
results = pd.concat([pd.read_csv(f) for f in results_files], ignore_index=True)

task_order = ["Suturing", "Needle Passing", "Knot Tying"]
task_labels = {
    "Suturing": "Suturing",
    "Needle_Passing": "Needle Passing",
    "Needle Passing": "Needle Passing",
    "Knot_Tying": "Knot Tying",
    "Knot Tying": "Knot Tying",
}

scratch = (
    results.loc[
        (results["kind"] == "Gesture") & (results["type"] == "from scratch"),
        ["task", "accuracy", "weighted_f1"],
    ]
    .assign(task=lambda df: df["task"].map(task_labels))
    .set_index("task")
    .reindex(task_order)
)

In [ ]:
# Phase->gesture transfer (Section 4.7, Tables 9-11) is commented out: its experiment code is not in this repository.
# # Results over five seeds, from Tables 9-11 of the paper.
# transfer = pd.DataFrame(
#     [
#         ("Suturing", 42, 0.8288, 0.8545, 0.8310),
#         ("Suturing", 257, 0.8192, 0.8512, 0.8287),
#         ("Suturing", 3121, 0.8400, 0.8337, 0.8242),
#         ("Suturing", 7576, 0.8369, 0.8409, 0.8274),
#         ("Suturing", 9438, 0.8350, 0.8398, 0.8443),
#         ("Needle Passing", 42, 0.7634, 0.8098, 0.7664),
#         ("Needle Passing", 257, 0.7808, 0.7853, 0.7821),
#         ("Needle Passing", 3121, 0.7263, 0.8164, 0.7840),
#         ("Needle Passing", 7576, 0.7761, 0.7928, 0.7576),
#         ("Needle Passing", 9438, 0.7579, 0.7972, 0.7653),
#         ("Knot Tying", 42, 0.9658, 0.8316, 0.8269),
#         ("Knot Tying", 257, 0.9584, 0.8524, 0.8418),
#         ("Knot Tying", 3121, 0.4618, 0.8188, 0.8564),
#         ("Knot Tying", 7576, 0.6059, 0.8442, 0.8600),
#         ("Knot Tying", 9438, 0.9563, 0.8415, 0.8722),
#     ],
#     columns=["task", "seed", "phase_accuracy", "phase_to_gesture_accuracy", "gesture_only_accuracy"],
# )

# transfer_summary = (
#     transfer.groupby("task")["phase_to_gesture_accuracy"]
#     .agg(mean="mean", min="min", max="max")
#     .reindex(task_order)
# )

In [ ]:
# Results of grs_train.ipynb
grs_results = (
    results.loc[results["kind"] == "Gesture + GRS"]
    .assign(task=lambda df: df["task"].map(task_labels))
    .set_index("task")
    .reindex(task_order)
)

grs = grs_results[["gesture_accuracy", "gesture_weighted_f1"]].set_axis(["accuracy", "f1"], axis=1)

In [ ]:
accuracy_plot = pd.DataFrame(
    {
        "Gesture from scratch": scratch["accuracy"],
        "Gesture by GRS model": grs["accuracy"],
    },
    index=task_order,
)

accuracy_errors = pd.DataFrame(0.0, index=task_order, columns=accuracy_plot.columns)

accuracy_plot

In [ ]:
accuracy_fig, ax = plt.subplots(figsize=(12.75, 7.2))

x = np.arange(len(task_order))
bar_width = 0.30
colors = ["#4C78A8", "#54A24B"]

for i, (label, color) in enumerate(zip(accuracy_plot.columns, colors)):
    offset = (i - 0.5) * bar_width
    yerr = accuracy_errors[label].to_numpy()
    bars = ax.bar(
        x + offset,
        accuracy_plot[label].to_numpy(),
        bar_width,
        label=label,
        color=color,
        edgecolor="#222222",
        linewidth=0.6,
        yerr=yerr if yerr.any() else None,
        capsize=4,
        error_kw={"elinewidth": 1.0, "capthick": 1.0, "ecolor": "#333333"},
    )
    ax.bar_label(bars, labels=[f"{v:.3f}" for v in accuracy_plot[label]], padding=3, fontsize=14)

ax.set_ylabel("Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(task_order)
ax.set_ylim(0.0, 1.0)
ax.legend(ncols=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.13))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.28)
ax.grid(axis="x", visible=False)

accuracy_fig.tight_layout()
plt.show()

In [ ]:
phase_scratch = (
    results.loc[
        (results["kind"] == "Phase") & (results["type"] == "from scratch"),
        ["task", "accuracy", "weighted_f1"],
    ]
    .assign(task=lambda df: df["task"].map(task_labels))
    .set_index("task")
    .reindex(task_order)
)

phase_transfer_types = ["full model fine-tuning", "frozen encoder", "linear probe"]
phase_transfer = (
    results.loc[
        (results["kind"] == "Phase") & (results["type"].isin(phase_transfer_types)),
        ["task", "type", "accuracy", "weighted_f1"],
    ]
    .assign(task=lambda df: df["task"].map(task_labels))
)

phase_transfer_summary = (
    phase_transfer.groupby("task")["accuracy"]
    .agg(mean="mean", min="min", max="max")
    .reindex(task_order)
)

phase_accuracy_plot = pd.DataFrame(
    {
        "Phase from scratch": phase_scratch["accuracy"],
        "Gesture->Phase transfer": phase_transfer_summary["mean"],
    },
    index=task_order,
)

phase_accuracy_mins = phase_accuracy_plot.copy()
phase_accuracy_mins["Gesture->Phase transfer"] = phase_transfer_summary["min"]

phase_accuracy_maxs = phase_accuracy_plot.copy()
phase_accuracy_maxs["Gesture->Phase transfer"] = phase_transfer_summary["max"]

phase_accuracy_plot


In [ ]:
phase_fig, ax = plt.subplots(figsize=(12.75, 7.2))

x = np.arange(len(task_order))
bar_width = 0.30
colors = ["#4C78A8", "#F58518"]

for i, (label, color) in enumerate(zip(phase_accuracy_plot.columns, colors)):
    offset = (i - 0.5) * bar_width
    values = phase_accuracy_plot[label].to_numpy()
    mins = phase_accuracy_mins[label].to_numpy()
    maxs = phase_accuracy_maxs[label].to_numpy()
    yerr = np.vstack([values - mins, maxs - values])
    bars = ax.bar(
        x + offset,
        values,
        bar_width,
        label=label,
        color=color,
        edgecolor="#222222",
        linewidth=0.6,
        yerr=yerr if yerr.any() else None,
        capsize=4,
        error_kw={"elinewidth": 1.0, "capthick": 1.0, "ecolor": "#333333"},
    )
    ax.bar_label(bars, labels=[f"{v:.3f}" for v in phase_accuracy_plot[label]], padding=3, fontsize=14)

ax.set_ylabel("Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(task_order)
ax.set_ylim(0.0, 1.0)
ax.legend(ncols=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.13))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.28)
ax.grid(axis="x", visible=False)

phase_fig.tight_layout()
plt.show()


In [ ]:
# Phase->gesture transfer (Section 4.7, Tables 9-11) is commented out: its experiment code is not in this repository.
# transfer_condition_summary = (
#     transfer.groupby("task")[["phase_accuracy", "phase_to_gesture_accuracy", "gesture_only_accuracy"]]
#     .agg(["mean", "min", "max"])
#     .reindex(task_order)
# )

# transfer_condition_plot = pd.DataFrame(
#     {
#         "Phase accuracy": transfer_condition_summary[("phase_accuracy", "mean")],
#         "Phase->Gesture accuracy": transfer_condition_summary[("phase_to_gesture_accuracy", "mean")],
#         "Gesture-only accuracy": transfer_condition_summary[("gesture_only_accuracy", "mean")],
#     },
#     index=task_order,
# )

# transfer_condition_mins = pd.DataFrame(
#     {
#         "Phase accuracy": transfer_condition_summary[("phase_accuracy", "min")],
#         "Phase->Gesture accuracy": transfer_condition_summary[("phase_to_gesture_accuracy", "min")],
#         "Gesture-only accuracy": transfer_condition_summary[("gesture_only_accuracy", "min")],
#     },
#     index=task_order,
# )

# transfer_condition_maxs = pd.DataFrame(
#     {
#         "Phase accuracy": transfer_condition_summary[("phase_accuracy", "max")],
#         "Phase->Gesture accuracy": transfer_condition_summary[("phase_to_gesture_accuracy", "max")],
#         "Gesture-only accuracy": transfer_condition_summary[("gesture_only_accuracy", "max")],
#     },
#     index=task_order,
# )

# transfer_condition_plot


In [ ]:
# Phase->gesture transfer (Section 4.7, Tables 9-11) is commented out: its experiment code is not in this repository.
# transfer_condition_fig, ax = plt.subplots(figsize=(12.75, 7.2))

# x = np.arange(len(task_order))
# bar_width = 0.24
# colors = ["#4C78A8", "#F58518", "#72B7B2"]

# for i, (label, color) in enumerate(zip(transfer_condition_plot.columns, colors)):
#     offset = (i - 1) * bar_width
#     values = transfer_condition_plot[label].to_numpy()
#     mins = transfer_condition_mins[label].to_numpy()
#     maxs = transfer_condition_maxs[label].to_numpy()
#     yerr = np.vstack([values - mins, maxs - values])
#     bars = ax.bar(
#         x + offset,
#         values,
#         bar_width,
#         label=label,
#         color=color,
#         edgecolor="#222222",
#         linewidth=0.6,
#         yerr=yerr,
#         capsize=4,
#         error_kw={"elinewidth": 1.0, "capthick": 1.0, "ecolor": "#333333"},
#     )
#     ax.bar_label(bars, labels=[f"{v:.3f}" for v in transfer_condition_plot[label]], padding=3, fontsize=14)

# ax.set_ylabel("Accuracy")
# ax.set_xticks(x)
# ax.set_xticklabels(task_order)
# ax.set_ylim(0.0, 1.0)
# ax.legend(ncols=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.13))
# ax.spines[["top", "right"]].set_visible(False)
# ax.grid(axis="y", alpha=0.28)
# ax.grid(axis="x", visible=False)

# transfer_condition_fig.tight_layout()
# plt.show()


In [ ]:
merged_comparison_plot = pd.DataFrame(
    {
        "Gesture from scratch": accuracy_plot["Gesture from scratch"],
        "Gesture by GRS model": accuracy_plot["Gesture by GRS model"],
        "Phase from scratch": phase_accuracy_plot["Phase from scratch"],
        "Gesture->Phase transfer": phase_accuracy_plot["Gesture->Phase transfer"],
    },
    index=task_order,
)

merged_comparison_mins = merged_comparison_plot.copy()
merged_comparison_mins["Gesture->Phase transfer"] = phase_accuracy_mins["Gesture->Phase transfer"]

merged_comparison_maxs = merged_comparison_plot.copy()
merged_comparison_maxs["Gesture->Phase transfer"] = phase_accuracy_maxs["Gesture->Phase transfer"]

merged_comparison_plot


In [ ]:
merged_comparison_fig, ax = plt.subplots(figsize=(13.8, 7.2))

x = np.arange(len(task_order))
bar_width = 0.18
colors = ["#4C78A8", "#54A24B", "#9D755D", "#F58518"]

for i, (label, color) in enumerate(zip(merged_comparison_plot.columns, colors)):
    offset = (i - 1.5) * bar_width
    values = merged_comparison_plot[label].to_numpy()
    mins = merged_comparison_mins[label].to_numpy()
    maxs = merged_comparison_maxs[label].to_numpy()
    yerr = np.vstack([values - mins, maxs - values])
    bars = ax.bar(
        x + offset,
        values,
        bar_width,
        label=label,
        color=color,
        edgecolor="#222222",
        linewidth=0.6,
        yerr=yerr if yerr.any() else None,
        capsize=4,
        error_kw={"elinewidth": 1.0, "capthick": 1.0, "ecolor": "#333333"},
    )
    ax.bar_label(bars, labels=[f"{v:.3f}" for v in merged_comparison_plot[label]], padding=3, fontsize=12)

ax.set_ylabel("Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(task_order)
ax.set_ylim(0.0, 1.0)
ax.legend(ncols=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.18))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.28)
ax.grid(axis="x", visible=False)

merged_comparison_fig.tight_layout()
plt.show()


## GRS Prediction Results

Task-average MAE, RMSE, and $R^2$ for GRS score prediction.

In [ ]:
grs_prediction_metrics = grs_results[["avg_grs_mae", "avg_grs_rmse", "avg_grs_r2"]].set_axis(["MAE", "RMSE", "R^2"], axis=1)
grs_prediction_metrics.index.name = "Task"

grs_prediction_metrics


In [ ]:
grs_prediction_fig, axes = plt.subplots(1, 3, figsize=(18.3, 6.3))

task_colors = ["#4C78A8", "#F58518", "#54A24B"]
task_tick_labels = ["Suturing", "Needle\nPassing", "Knot\nTying"]
metric_settings = {
    "MAE": {"ylim": None, "title": "MAE $\\downarrow$"},
    "RMSE": {"ylim": None, "title": "RMSE $\\downarrow$"},
    "R^2": {"ylim": (0.0, 1.0), "title": "$R^2$ $\\uparrow$"},
}

for ax, (metric, settings) in zip(axes, metric_settings.items()):
    values = grs_prediction_metrics[metric]
    bars = ax.bar(
        np.arange(len(values)),
        values,
        color=task_colors,
        edgecolor="#222222",
        linewidth=0.6,
    )
    ax.bar_label(bars, labels=[f"{value:.4f}" for value in values], padding=3, fontsize=14)
    ax.set_title(settings["title"])
    ax.set_ylabel(metric)
    ax.set_xticks(np.arange(len(values)))
    ax.set_xticklabels(task_tick_labels)
    ax.set_ylim(*(settings["ylim"] or (0.0, values.max() * 1.15)))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.28)
    ax.grid(axis="x", visible=False)

grs_prediction_fig.tight_layout()
plt.show()


In [ ]:
output_dir = PATHS['images_dir'] / "plots"
output_dir.mkdir(parents=True, exist_ok=True)

figures_to_save = {
    "gesture_accuracy": accuracy_fig,
    "phase_accuracy": phase_fig,
    # "phase_to_gesture_condition": transfer_condition_fig,
    "gesture_phase_merged_comparison": merged_comparison_fig,
    "grs_prediction_task_averages": grs_prediction_fig,
}

for name, fig in figures_to_save.items():
    fig.savefig(output_dir / f"{name}.png", bbox_inches="tight")
    fig.savefig(output_dir / f"{name}.pdf", bbox_inches="tight")
